In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import json
import cv2
from pathlib import Path
from transnetv2_pytorch import TransNetV2

DATA_DIR = Path("/kaggle/input/competitions/multi-lingual-video-fragment-retrieval-challenge/video-rag")
VIDEO_DIR = DATA_DIR / "videos"
OUTPUT_FILE = Path("shot_boundaries.json")
BATCH_SAVE = 10

model = TransNetV2(device="cuda")

video_files = sorted(
    list(VIDEO_DIR.glob("*.mp4")) + list(VIDEO_DIR.glob("*.mkv")) +
    list(VIDEO_DIR.glob("*.webm")) + list(VIDEO_DIR.glob("*.opus"))
)



In [ ]:
# Загружаем уже обработанные (resume)
if OUTPUT_FILE.exists():
    with open(OUTPUT_FILE, encoding="utf-8") as f:
        results = json.load(f)
    done = {r["video_id"] for r in results}
    print(f"Уже обработано: {len(done)}, осталось: {len(video_files) - len(done)}")
else:
    results = []
    done = set()

for i, video_path in enumerate(video_files):
    video_name = video_path.stem
    if video_name in done:
        continue

    print(f"[{len(done)+1}/{len(video_files)}] {video_name}")

    try:
        cap = cv2.VideoCapture(str(video_path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
        cap.release()

        video_frames, single_frame_predictions, _ = model.predict_video(str(video_path))
        predictions = single_frame_predictions.numpy() if hasattr(single_frame_predictions, 'numpy') else single_frame_predictions
        scenes_frames = model.predictions_to_scenes(predictions)

        scenes = []
        for idx, (start_frame, end_frame) in enumerate(scenes_frames):
            start_sec = round(int(start_frame) / fps, 3)
            end_sec = round(int(end_frame) / fps, 3)
            scenes.append({
                "scene_idx": idx,
                "start": start_sec,
                "end": end_sec,
                "description": None,
                "description_en": None,
                "keyframe_time": round((start_sec + end_sec) / 2, 3),
                "keyframe_path": None,
            })

        results.append({
            "video_id": video_name,
            "audio_key": f"videos/{video_path.name}",
            "scenes": scenes,
        })
        done.add(video_name)
        print(f"  → {len(scenes)} шотов")

    except Exception as e:
        print(f"  [ОШИБКА] {video_name}: {e}")

    # Сохраняем каждые 10
    if len(done) % BATCH_SAVE == 0:
        with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"  [сохранено {len(done)} видео]")

# Финальное сохранение
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"\nГотово! {len(results)} видео → {OUTPUT_FILE}")
